# Example 1: Agentic Customer Support Triage with MCP Tool
Companion notebook for **Example 1** of Chapter 6.

A compact **classify -> gather evidence -> route** pattern: classify the incoming request, fetch customer context through MCP tools, and route low-confidence cases to human review. Everything needed to run this end to end -- including a small MCP server -- is self-contained in this one notebook (unlike Chapter 5's notebooks, nothing here depends on another notebook having run first).

In [1]:
from typing import Literal, TypedDict


class SupportState(TypedDict, total=False):
    customer_id: str
    user_message: str
    triage_label: Literal["billing", "technical", "account"]
    confidence: float
    evidence: list[str]
    account_context: dict
    recent_tickets: list[dict]
    next_action: str

**Node 1 - `classify_issue`**: a rule-based stand-in for an LLM classifier. Keeping `evidence` in state means later nodes (and a human reviewer) can see *why* a route was chosen, not just *what* was chosen.

In [2]:
def classify_issue(state: SupportState) -> SupportState:
    text = state["user_message"].lower()

    if "charged" in text or "invoice" in text:
        return {
            "triage_label": "billing",
            "confidence": 0.91,
            "evidence": ["mentions charge or invoice"],
        }
    if "error" in text or "crash" in text:
        return {
            "triage_label": "technical",
            "confidence": 0.86,
            "evidence": ["mentions crash or error"],
        }
    return {
        "triage_label": "account",
        "confidence": 0.62,
        "evidence": ["defaulted to account-related category"],
    }

## A small MCP server for the support tools
`get_customer_account` and `get_recent_tickets` are the two approved business tools this workflow is allowed to call. In a real system these would wrap a CRM and a ticketing system; here they return deterministic mock data so the notebook needs no API keys and no network access.

In [3]:
%%writefile support_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("SupportServer")

_ACCOUNTS = {
    "CUST-1001": {"customer_id": "CUST-1001", "name": "Asha Mehta", "plan": "pro", "billing_status": "current"},
    "CUST-1002": {"customer_id": "CUST-1002", "name": "Rohan Iyer", "plan": "starter", "billing_status": "past_due"},
}

_TICKETS = {
    "CUST-1001": [
        {"ticket_id": "T-501", "status": "open", "category": "technical"},
        {"ticket_id": "T-477", "status": "resolved", "category": "billing"},
    ],
    "CUST-1002": [
        {"ticket_id": "T-610", "status": "open", "category": "billing"},
    ],
}


@mcp.tool()
def get_customer_account(customer_id: str) -> dict:
    """Return the account profile for a customer."""
    return _ACCOUNTS.get(customer_id, {"customer_id": customer_id, "plan": "unknown", "billing_status": "unknown"})


@mcp.tool()
def get_recent_tickets(customer_id: str) -> list[dict]:
    """Return recent support tickets for a customer."""
    return _TICKETS.get(customer_id, [])


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing support_server.py


## Node 2 - `fetch_context`
MCP gives the app a standard way to call approved business tools instead of a bespoke connector per system. `call_tool` below is a small helper that unwraps a FastMCP tool result into a plain Python value -- the same unwrapping shape you'll see reused in every notebook in this chapter.

In [4]:
import sys, os

# Jupyter's stdout/stderr streams don't expose a real OS file descriptor, which crashes
# the MCP SDK's Windows subprocess-spawning fallback (it isn't needed outside Jupyter).
if os.name == "nt":
    try:
        sys.stderr.fileno()
    except Exception:
        sys.stderr = os.fdopen(os.dup(2), "w")

import asyncio
import json
import sys
from pathlib import Path
from typing import Any

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langgraph.graph import StateGraph, START, END

async def call_tool(session: ClientSession, name: str, arguments: dict) -> Any:
    """Call an MCP tool and unwrap its result into a plain Python value."""
    result = await session.call_tool(name, arguments)
    if result.structuredContent is not None:
        content = result.structuredContent
        return content.get("result", content) if isinstance(content, dict) else content
    text = result.content[0].text
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text

In [5]:
async def fetch_context(state: SupportState, mcp_session: ClientSession) -> SupportState:
    account = await call_tool(mcp_session, "get_customer_account", {"customer_id": state["customer_id"]})
    tickets = await call_tool(mcp_session, "get_recent_tickets", {"customer_id": state["customer_id"]})
    return {
        "account_context": account,
        "recent_tickets": tickets,
        "evidence": state["evidence"] + ["fetched account and ticket history"],
    }

**Node 3 - `route_case`**: low confidence always wins, regardless of label -- a common LangGraph pattern for controlled escalation.

In [6]:
def route_case(state: SupportState) -> SupportState:
    if state["confidence"] < 0.75:
        return {"next_action": "human_review"}
    if state["triage_label"] == "billing":
        return {"next_action": "billing_agent"}
    if state["triage_label"] == "technical":
        return {"next_action": "tech_agent"}
    return {"next_action": "account_agent"}

## Wiring the graph: `classify -> fetch_context -> route`

In [7]:
def build_graph(mcp_session: ClientSession):
    graph = StateGraph(SupportState)

    async def fetch_context_node(state: SupportState) -> SupportState:
        return await fetch_context(state, mcp_session)

    graph.add_node("classify_issue", classify_issue)
    graph.add_node("fetch_context", fetch_context_node)
    graph.add_node("route_case", route_case)

    graph.add_edge(START, "classify_issue")
    graph.add_edge("classify_issue", "fetch_context")
    graph.add_edge("fetch_context", "route_case")
    graph.add_edge("route_case", END)
    return graph.compile()

In [8]:
server_path = Path.cwd() / "support_server.py"
params = StdioServerParameters(command=sys.executable, args=[str(server_path)])

async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        app = build_graph(session)
        result = await app.ainvoke({
            "customer_id": "CUST-1001",
            "user_message": "I was charged twice on my last invoice and need help.",
        })

print(f"label={result['triage_label']}  confidence={result['confidence']}  next_action={result['next_action']}")
print("evidence:", result["evidence"])
print("account plan:", result["account_context"].get("plan"))
print("open tickets:", [t for t in result["recent_tickets"] if t["status"] == "open"])

label=billing  confidence=0.91  next_action=billing_agent
evidence: ['mentions charge or invoice', 'fetched account and ticket history']
account plan: pro
open tickets: [{'ticket_id': 'T-501', 'status': 'open', 'category': 'technical'}]
